# Module install and load

In [1]:
!pip install scikit-image


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\Patrick\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install pillow numpy pandas


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\Patrick\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Module import

In [3]:
# 1. Import Required Libraries
import os
from PIL import Image, ImageStat
import numpy as np
import pandas as pd
from skimage.transform import radon
from skimage.feature import canny
import scipy.stats
import matplotlib.pyplot as plt
import re
from pathlib import Path


# List image files

In [4]:


image_folder = Path(r"C:\Users\Patrick\Gutbrain_R\data\food")
image_files = [f.name for f in image_folder.iterdir() if f.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}]
print(f"Found {len(image_files)} images.")

Found 70 images.


# Functions

## Function to calculate visual properties per image

In [5]:


## 3. Calculate visual properties for each image
def calculate_image_properties(image_path):
    with Image.open(image_path) as img:
        img = img.convert('RGB')
        np_img = np.array(img)
        size = img.size

        # Mask: True for object pixels (not black)
        custom_masks = {
            "HiSaltPretzels.bmp": [0, 4],
            "HiSweetm&mamandes.bmp": [0, 1],
            "HiSweetReese.bmp": [0, 2]
        }

        filename = os.path.basename(image_path)
        gray_img = np.dot(np_img[...,:3], [0.2989, 0.5870, 0.1140]).astype(int)

        if filename in custom_masks:
            # Mask: True for object pixels (not background gray levels)
            mask = ~np.isin(gray_img, custom_masks[filename])
        else:
            # Default: True for object pixels (not almost black)
            mask = np.any(np_img > 1, axis=-1)

        object_pixels = np_img[mask]
        num_object_pixels = object_pixels.shape[0]
        total_pixels = size[0] * size[1]

        if num_object_pixels > 0:
            mean_color = np.round(object_pixels.mean(axis=0), 2)

            gray_object_pixels = np.dot(object_pixels[...,:3], [0.2989, 0.5870, 0.1140])
            brightness = np.round(gray_object_pixels.mean(), 2)
            contrast = np.round(gray_object_pixels.std(), 2)

            
        else:
            mean_color = [np.nan, np.nan, np.nan]
            brightness = np.nan
            contrast = np.nan


        return {
            'filename': os.path.basename(image_path),
            'width': size[0],
            'height': size[1],
            'mean_color_r': mean_color[0],
            'mean_color_g': mean_color[1],
            'mean_color_b': mean_color[2],
            'brightness': brightness,
            'contrast': contrast,
            'num_object_pixels': num_object_pixels

        }
    


# Compute visual properties and analyze them

## Run function on images and describe output table

In [6]:
results = [calculate_image_properties(os.path.join(image_folder, f)) for f in image_files]
df = pd.DataFrame(results)

numeric_cols = df.select_dtypes(include=[np.number]).columns
zscore_df = df[numeric_cols].apply(scipy.stats.zscore, nan_policy='omit')
zscore_df = zscore_df.add_suffix('_zscore')

# Concatenate z-scores to the original DataFrame
df = pd.concat([df, zscore_df], axis=1)

## 4. Display summary statistics
df.describe()

,width,height,mean_color_r,mean_color_g,mean_color_b,brightness,contrast,num_object_pixels,width_zscore,height_zscore,mean_color_r_zscore,mean_color_g_zscore,mean_color_b_zscore,brightness_zscore,contrast_zscore,num_object_pixels_zscore
count,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,7.000000e+01,7.000000e+01,7.000000e+01,7.000000e+01,7.000000e+01,7.000000e+01,7.000000e+01,7.000000e+01
mean,1308.757143,731.657143,145.638571,109.169571,85.481571,117.358000,71.689571,66581.928571,6.873867e-15,1.488492e-15,1.427430e-16,1.427430e-17,1.586033e-17,-6.312411e-16,1.532504e-16,4.504333e-16
std,15.249078,8.707627,36.461072,34.311427,35.066566,29.826485,14.216324,14488.454949,1.007220e+00,1.007220e+00,1.007220e+00,1.007220e+00,1.007220e+00,1.007220e+00,1.007220e+00,1.007220e+00
min,1192.000000,666.000000,48.240000,25.090000,10.780000,46.000000,33.250000,42046.000000,-7.711953e+00,-7.594630e+00,-2.690591e+00,-2.468176e+00,-2.145660e+00,-2.409712e+00,-2.723427e+00,-1.705709e+00
25%,1309.000000,731.000000,123.700000,87.745000,60.122500,96.865000,60.987500,56019.500000,1.604101e-02,-7.601240e-02,-6.060429e-01,-6.289235e-01,-7.283910e-01,-6.920348e-01,-7.582371e-01,-7.342876e-01
50%,1311.500000,733.000000,143.815000,106.120000,83.650000,115.070000,72.505000,65290.500000,1.811691e-01,1.553297e-01,-5.037532e-02,-8.952091e-02,-5.260840e-02,-7.726422e-02,5.777276e-02,-8.977859e-02
75%,1313.000000,734.750000,170.042500,133.147500,109.652500,134.570000,81.395000,76548.250000,2.802459e-01,3.577540e-01,6.741473e-01,7.038779e-01,6.942639e-01,5.812376e-01,6.876254e-01,6.928469e-01
max,1316.000000,739.000000,247.820000,194.820000,164.530000,194.780000,106.420000,114790.000000,4.783996e-01,8.493559e-01,2.822715e+00,2.514289e+00,2.270516e+00,2.614489e+00,2.460636e+00,3.351368e+00


## Show the table of visual properties

In [7]:
df.head(50)

,filename,width,height,mean_color_r,mean_color_g,mean_color_b,brightness,contrast,num_object_pixels,width_zscore,height_zscore,mean_color_r_zscore,mean_color_g_zscore,mean_color_b_zscore,brightness_zscore,contrast_zscore,num_object_pixels_zscore
0,HiSaltAmandes.jpg,1308,738,124.08,139.12,140.76,134.80,80.07,64814,-0.050010,0.733685,-0.595546,0.879202,1.587768,0.589005,0.593750,-0.122904
1,HiSaltcheetos.bmp,1309,732,148.15,122.77,81.73,125.66,63.12,63844,0.016041,0.039659,0.069377,0.399244,-0.107757,0.280353,-0.607150,-0.190338
2,HiSaltCrispers.JPG,1314,729,165.32,105.94,86.52,121.46,98.54,67737,0.346297,-0.307354,0.543690,-0.094805,0.029827,0.138522,1.902341,0.080299
3,HiSaltCrispyminis.bmp,1309,732,75.75,76.62,78.64,76.58,59.93,79327,0.016041,0.039659,-1.930640,-0.955501,-0.196511,-1.377046,-0.833160,0.886022
4,HiSaltDoritos.bmp,1309,730,183.95,86.15,58.27,112.19,63.17,69901,0.016041,-0.191683,1.058336,-0.675745,-0.781601,-0.174520,-0.603608,0.230738
5,HiSaltFritos.bmp,1315,737,184.93,103.93,50.25,122.01,61.76,55823,0.412348,0.618014,1.085408,-0.153809,-1.011960,0.157095,-0.703506,-0.747948
6,HiSaltGoldfish2.bmp,1313,732,123.97,110.47,88.48,111.99,87.24,110235,0.280246,0.039659,-0.598584,0.038174,0.086124,-0.181274,1.101741,3.034710
7,HiSaltJuliennes.JPG,1311,731,171.66,135.85,64.76,138.44,83.61,68369,0.148143,-0.076012,0.718830,0.783210,-0.595188,0.711925,0.844557,0.124235
8,HiSaltLays.bmp,1314,737,185.40,146.39,53.71,147.47,55.12,78561,0.346297,0.618014,1.098391,1.092615,-0.912578,1.016862,-1.173947,0.832771
9,HiSaltLaysketchup.JPG,1312,734,183.53,105.69,93.17,127.52,85.17,83255,0.214195,0.271001,1.046733,-0.102144,0.220835,0.343164,0.955082,1.159092


# Merge with updated pricing

## upload pricing csv to df

In [8]:
pricing_path = Path(r"C:\Users\Patrick\Gutbrain_R\data\kcal_real_v3.csv")
pricing_df = pd.read_csv(pricing_path, sep=",")

## Clean the pricing df

In [9]:
# Drop columns only if they exist in pricing_df
cols_to_keep = ['picture', 'Moyenne Densite'] + list(pricing_df.columns[-5:])
pricing_df = pricing_df[cols_to_keep]
pricing_df.head()

pricing_df['picture'] = pricing_df['picture'].str.replace(r'^Food/', '', regex=True)


## merge pricing and visual properties df

In [10]:
pricing_df['picture'] = (
    pricing_df['picture']
    .astype(str)
    .str.lower()
    .str.replace(r'^food/', '', regex=True)
    .str.replace(r'\..*$', '', regex=True)
)

df['filename'] = (
    df['filename']
    .astype(str)
    .str.lower()
    .str.replace(r'\..*$', '', regex=True)
)

merged_df = pd.merge(
    pricing_df,
    df,
    left_on='picture',
    right_on='filename',
    how='inner'
)

In [11]:
merged_df

,picture,Moyenne Densite,Grammes montrés (Selon l'image),Prix,Prix/gramme,Source prix,Commentaire.1,filename,width,height,...,contrast,num_object_pixels,width_zscore,height_zscore,mean_color_r_zscore,mean_color_g_zscore,mean_color_b_zscore,brightness_zscore,contrast_zscore,num_object_pixels_zscore
0,lonectarine,0.440588,153.000,1.3464,0.008800,https://www.superc.ca/allees/fruits-et-legumes...,136g pour 1 poire complète selon alimentation ...,lonectarine,1310,730,...,75.24,48899,0.082092,-0.191683,0.847008,-1.268134,-1.576943,-0.758054,0.251546,-1.229296
1,loapplesauce,0.741786,109.000,0.6090,0.005587,https://www.instacart.ca/products/17884558?ret...,"1 pack de 6x104g =3,49$ et une cuillère à thé ...",loapplesauce,1280,720,...,63.35,80210,-1.899445,-1.348394,1.236790,2.463211,1.241080,2.281523,-0.590855,0.947407
2,lowelchs,3.107143,22.000,0.3390,0.015409,https://www.maxi.ca/en/fruit-snacks-mixed-frui...,"boite de 28 sachets de 22g, prix boite 9,50$",lowelchs,1313,735,...,72.94,75917,0.280246,0.386672,-1.796108,-0.970765,-0.234426,-1.343276,0.088592,0.648963
3,loraisins,0.690000,80.000,0.6200,0.007750,https://www.maxi.ca/fr/raisins-verts-extra-gro...,pour 16 raisins (selon alimentation canada: 10...,loraisins,1280,720,...,58.37,79493,-1.899445,-1.348394,-0.154934,1.104357,-1.577517,0.477904,-0.943686,0.897562
4,locheesestrings,2.857143,42.000,0.8100,0.019286,https://www.maxi.ca/en/marble-cheestrings/p/20...,pour 2 ficellos (21g x 16 unités = 336g) pris ...,locheesestrings,1312,734,...,58.61,114790,0.214195,0.271001,-2.690591,-1.842616,-1.360084,-2.409712,-0.926682,3.351368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,hisaltpartymix,5.100000,255.000,3.5700,0.014000,https://www.maxi.ca/en/party-original-mix/p/21...,"Mélange hétérogène, 1 1/4 tasse = 50g selon fi...",hisaltpartymix,1310,730,...,83.55,80207,0.082092,-0.191683,0.013023,-0.570947,-0.086789,-0.392332,0.840306,0.947199
66,hisweetm&mamandes,5.288010,107.440,5.6500,0.052587,https://www.jeancoutu.com/magasiner/categories...,Semble être un plus petit format que celui des...,hisweetm&mamandes,1310,730,...,89.24,89787,0.082092,-0.191683,0.203080,0.713051,0.363302,0.604201,1.243440,1.613189
67,hisaltjuliennes,5.400000,94.700,2.5000,0.026399,https://www.walmart.ca/en/ip/hostess-hickory-s...,"Prix pour 1 sac de 90g et pour 4,7g de chips h...",hisaltjuliennes,1311,731,...,83.61,68369,0.148143,-0.076012,0.718830,0.783210,-0.595188,0.711925,0.844557,0.124235
68,hisweethersheyamandes,5.190976,43.000,1.1800,0.027442,https://www.walmart.ca/en/ip/hersheys-creamy-m...,pri x pour 43g comme sur photo.,hisweethersheyamandes,1313,732,...,76.06,53585,0.280246,0.039659,-0.751348,-0.739152,-0.520508,-0.843490,0.309643,-0.903531


## Open the df_grouped_by_picture

In [12]:
df_grouped_by_picture = pd.read_csv('C:\\Users\\Patrick\\Gutbrain_R\\data\\df_grouped_by_picture.csv', sep=',')


## set up for the merge

In [13]:
# Standardize 'picture' columns for merging

def clean_picture(s):
    s = s.lower()
    s = re.sub(r'^(lo|hisweet|hisalt)', '', s)
    s = re.sub(r'\..*$', '', s)
    s = s.strip('_')
    if s == 'juliennes':
        s = 'julienne'
    return s

merged_df['picture_clean'] = merged_df['picture'].apply(clean_picture)
df_grouped_by_picture['picture_clean'] = df_grouped_by_picture['picture'].apply(clean_picture)

# Use left join to keep all rows from df_grouped_by_picture
final_merged_df = pd.merge(df_grouped_by_picture, merged_df, on='picture_clean', how='left')


# Use the correct column name for missing
missing = final_merged_df[final_merged_df.isnull().any(axis=1)]['picture_x']


C:\Users\Patrick\AppData\Local\Temp\ipykernel_36560\2350397771.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_grouped_by_picture['picture_clean'] = df_grouped_by_picture['picture'].apply(clean_picture)


In [14]:
print("Cleaned names in df_grouped_by_picture:")
print(sorted(df_grouped_by_picture['picture_clean'].unique()))
print("\nCleaned names in merged_df:")
print(sorted(merged_df['picture_clean'].unique()))

Cleaned names in df_grouped_by_picture:
['activia', 'aero', 'amandes', 'apple', 'applesauce', 'babybel', 'banana', 'brownies', 'caramilk', 'carottes', 'cheesestrings', 'cheetos', 'crackerjack', 'crispers', 'crispyminis', 'cucumber', 'dairymilk', 'dipps', 'doritos', 'ferrerorocher', 'fritos', 'goldfish2', 'handisnacks', 'hersheyamandes', 'hersheykisses', 'julienne', 'kitkat', 'kiwi', 'lays', 'laysketchup', 'lindt70cacao', 'm&mamandes', 'mandm', 'mars', 'maynards', 'maynardspeaches', 'melangerandonneur', 'melimelo', 'missvickies', 'munchos', 'nectarine', 'oatmeal', 'orange', 'oreo', 'partymix', 'pear', 'plum', 'popcorn', 'poptarts', 'pouding', 'praeventia', 'pretzels', 'pringles', 'raisins', 'redpepper', 'reese', 'ritzbits', 'ruffles', 'rufflesalldressed', 'rufflesregular', 'skittles', 'snickers', 'sunchips', 'toblerone', 'tomatescerises', 'tostitos', 'twix', 'verrycherry', 'welchs', 'werthers']

Cleaned names in merged_df:
['activia', 'aero', 'amandes', 'apple', 'applesauce', 'babybel',

In [15]:
print(final_merged_df.describe())
print("\nNaN count per column:")
print(final_merged_df.isna().sum())

          session        sexe       sexeF        t2d    type_chx  tx_t2d___1  \
count  280.000000  280.000000  280.000000  70.000000  280.000000   70.000000   
mean     2.500000    1.765028    0.765028   1.772421    1.806845    0.065385   
std      1.120036    0.048688    0.048688   0.048086    0.095772    0.025548   
min      1.000000    1.583333    0.583333   1.639535    1.475000    0.000000   
25%      1.750000    1.742772    0.742772   1.746521    1.750000    0.056207   
50%      2.500000    1.765739    0.765739   1.769615    1.808806    0.070178   
75%      3.250000    1.791043    0.791043   1.799468    1.859191    0.076442   
max      4.000000    1.894118    0.894118   1.909091    2.178571    0.125000   

       tx_t2d___2  tx_t2d___3      date_irm         age  ...    contrast  \
count   70.000000        70.0    280.000000  280.000000  ...  280.000000   
mean     0.227579         0.0  43657.096812   45.213174  ...   71.689571   
std      0.048086         0.0    305.444339    1.67

# save df of stimuli properties

In [16]:
final_merged_df.to_csv('C:\\Users\\Patrick\\Gutbrain_R\\data\\gutbrain_food_final_merged_df.tsv', sep='\t', index=False)
print("Saved to C:\\Users\\Patrick\\Gutbrain_R\\data\\gutbrain_food_final_merged_df.tsv")

Saved to C:\Users\Patrick\Gutbrain_R\data\gutbrain_food_final_merged_df.tsv
